In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor,AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.linear_model import LinearRegression,Ridge,Lasso
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error
from sklearn.model_selection import RandomizedSearchCV
import warnings

In [4]:
!pip install catboost
!pip install xgboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 45.9 MB/s  0:00:026m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [catboost]1/2 [catboost]

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip


In [5]:
from catboost import CatBoostRegressor
from xgboost import XGBRegressor

In [6]:
df=pd.read_csv('stud.csv')

In [7]:
df.head(5)

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


In [8]:
x=df.drop(columns=['math_score'],axis=1)

In [9]:
x.head()

,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,reading_score,writing_score
0,female,group B,bachelor's degree,standard,none,72,74
1,female,group C,some college,standard,completed,90,88
2,female,group B,master's degree,standard,none,95,93
3,male,group A,associate's degree,free/reduced,none,57,44
4,male,group C,some college,standard,none,78,75


In [10]:
y=df['math_score']

In [11]:
y

0      72
1      69
2      90
3      47
4      76
       ..
995    88
996    62
997    59
998    68
999    77
Name: math_score, Length: 1000, dtype: int64

In [17]:
num_features = x.select_dtypes(exclude=["object"]).columns
cat_features = x.select_dtypes(include=["object"]).columns

from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

numeric_transformer = StandardScaler()
oh_transformer = OneHotEncoder()

preprocessor = ColumnTransformer(
    [
        ("OneHotEncoder", oh_transformer, cat_features),
        ("StandardScaler", numeric_transformer, num_features),
    ]
)

In [18]:
x=preprocessor.fit_transform(x)

In [19]:
x

array([[ 1.        ,  0.        ,  0.        , ...,  1.        ,
         0.19399858,  0.39149181],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         1.42747598,  1.31326868],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         1.77010859,  1.64247471],
       ...,
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.12547206, -0.20107904],
       [ 1.        ,  0.        ,  0.        , ...,  0.        ,
         0.60515772,  0.58901542],
       [ 1.        ,  0.        ,  0.        , ...,  1.        ,
         1.15336989,  1.18158627]], shape=(1000, 19))

In [20]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
x_train.shape, x_test.shape

((800, 19), (200, 19))

In [21]:
def evaluate_model(true,predicted):
    mae= mean_absolute_error(true,predicted)
    mse= mean_squared_error(true,predicted)
    rmse= np.sqrt(mean_squared_error(true,predicted))
    r2_score= r2_score(true,predicted)
    return mae,rmse,r2_score

In [30]:
models = {
    "LinearRegression": LinearRegression(),
    "Lasso": Lasso(),
    "Ridge": Ridge(),
    "KNeighborsRegressor": KNeighborsRegressor(),
    "DecisionTreeRegressor": DecisionTreeRegressor(),
    "RandomForestRegressor": RandomForestRegressor(),
    "XGBRegressor": XGBRegressor(),
    "CatBoostRegressor": CatBoostRegressor(verbose=0),
    "AdaBoostRegressor": AdaBoostRegressor()
}

model_list = []
r2_list = []

for name, model in models.items():
    model.fit(x_train, y_train)

    y_train_pred = model.predict(x_train)
    y_test_pred = model.predict(x_test)

    # Training metrics
    train_r2 = r2_score(y_train, y_train_pred)
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_mse = mean_squared_error(y_train, y_train_pred)

    # Testing metrics
    test_r2 = r2_score(y_test, y_test_pred)
    test_mae = mean_absolute_error(y_test, y_test_pred)
    test_mse = mean_squared_error(y_test, y_test_pred)

    print("\n" + "="*50)
    print(f"{name}")
    print("="*50)

    print("Model performance for Training set")
    print(f"R2 Score: {train_r2:.4f}")
    print(f"MAE: {train_mae:.4f}")
    print(f"MSE: {train_mse:.4f}")

    print("\nModel performance for Testing set")
    print(f"R2 Score: {test_r2:.4f}")
    print(f"MAE: {test_mae:.4f}")
    print(f"MSE: {test_mse:.4f}")


LinearRegression
Model performance for Training set
R2 Score: 0.8743
MAE: 4.2667
MSE: 28.3349

Model performance for Testing set
R2 Score: 0.8804
MAE: 4.2148
MSE: 29.0952

Lasso
Model performance for Training set
R2 Score: 0.8071
MAE: 5.2063
MSE: 43.4784

Model performance for Testing set
R2 Score: 0.8253
MAE: 5.1579
MSE: 42.5064

Ridge
Model performance for Training set
R2 Score: 0.8743
MAE: 4.2650
MSE: 28.3378

Model performance for Testing set
R2 Score: 0.8806
MAE: 4.2111
MSE: 29.0563

KNeighborsRegressor
Model performance for Training set
R2 Score: 0.8558
MAE: 4.5070
MSE: 32.5070

Model performance for Testing set
R2 Score: 0.7822
MAE: 5.6390
MSE: 53.0010

DecisionTreeRegressor
Model performance for Training set
R2 Score: 0.9997
MAE: 0.0187
MSE: 0.0781

Model performance for Testing set
R2 Score: 0.7319
MAE: 6.3800
MSE: 65.2400

RandomForestRegressor
Model performance for Training set
R2 Score: 0.9766
MAE: 1.8245
MSE: 5.2786

Model performance for Testing set
R2 Score: 0.8550
MAE:

In [33]:
for name, model in models.items():
    model.fit(x_train, y_train)

    y_test_pred = model.predict(x_test)

    test_r2 = r2_score(y_test, y_test_pred)

    model_list.append(name)
    r2_list.append(test_r2)

In [34]:
pd.DataFrame(list(zip(model_list, r2_list)), columns=["Model Name", "R2 Score"]).sort_values(by="R2 Score", ascending=False)

,Model Name,R2 Score
2,Ridge,0.880593
0,LinearRegression,0.880433
7,CatBoostRegressor,0.851632
5,RandomForestRegressor,0.850433
8,AdaBoostRegressor,0.848258
6,XGBRegressor,0.827797
1,Lasso,0.825320
3,KNeighborsRegressor,0.782192
4,DecisionTreeRegressor,0.734423
